In [ ]:
from app.services.pipeline import coletar_dados

from time import perf_counter

username = "felipe.cruz"
password = "#Gladoscruz.9851"
analise = "compra por necessidade"

dfs = coletar_dados(username, password, analise)

In [ ]:
import pickle

with open("snapshot_dfs.pkl", "wb") as f:
    pickle.dump(dfs, f)

print("Snapshot salvo.")



In [2]:
import pickle

dfs = {}
with open("snapshot_dfs.pkl", "rb") as f:
    dfs = pickle.load(f)

In [ ]:
import pandas as pd
import networkx as nx

print(f"{dfs.keys()}\n")


def sanitizar_dataframe(df, limite=0.8):
    df = df.copy()

    for col in df.columns:
        serie = df[col].astype(str).str.strip()

        tentativa_data = pd.to_datetime(
            serie, errors="coerce", dayfirst=True, format="%d/%m/%Y"
        )
        if tentativa_data.notna().mean() > limite:
            df[col] = tentativa_data
            continue

        serie_num = serie.str.replace(".", "", regex=False).str.replace(
            ",", ".", regex=False
        )
        tentativa_num = pd.to_numeric(serie_num, errors="coerce")
        if tentativa_num.notna().mean() > limite:
            df[col] = tentativa_num
            continue

        df[col] = serie.replace({"": None})

    return df


def calc_data(dfs):

    ## Ajuste Ordens ##
    ordens = sanitizar_dataframe(dfs.get("ordens"))
    ordens = ordens[
        [
            "Cliente",
            "Fábrica",
            "Ordem",
            "Pedido",
            "Item",
            "Saldo",
            "Representante",
            "Entrega Pedido",
            "Data Abertura",
        ]
    ]
    ordens = ordens.rename(columns={"Ordem": "Ordem Prod", "Saldo": "Saldo Prod"})

    colunas = ["Entrega Pedido", "Data Abertura"]
    for col in colunas:
        # Remove o ponto e garante que a coluna seja tratada como string
        ordens[col] = (
            ordens[col]
            .astype(str)
            .str.strip()
            .str.replace(r"[^\d]", "", regex=True)  # remove tudo que não for número
            .pipe(lambda s: pd.to_datetime(s, format="%d%m%Y", errors="coerce"))
        )
    ## Ajuste Consumo ##
    consumo = sanitizar_dataframe(dfs.get("cons"))

    consumo["Item"] = consumo["Item"].str.split("-").str[0].str.strip()
    print(consumo.columns)
    consumo = consumo[["Item", "Baixa", "Consumo", "Local Prod.", "OP", "Familia"]]
    consumo = consumo.rename(columns={"OP": "Ordem Cons"})

    ######## Item Pai ##########

    consumo["item_pai"] = consumo["Ordem Cons"].map(
        ordens.set_index("Ordem Prod")["Item"]
    )
    ######## Importa dados ##########
    consumo = consumo.merge(ordens[["Item", "Ordem Prod"]], on="Item", how="left")

    consumo = consumo.merge(
        ordens[
            [
                "Cliente",
                "Fábrica",
                "Ordem Prod",
                "Pedido",
                "Saldo Prod",
                "Representante",
                "Entrega Pedido",
                "Data Abertura",
            ]
        ].rename(columns={"Ordem Prod": "Ordem", "Saldo Prod": "Saldo"}),
        left_on="Ordem Cons",
        right_on="Ordem",
        how="left",
    )

    ######## Ordena as colunas ##########

    consumo = consumo[
        [
            "Ordem Prod",
            "Item",
            "Consumo",
            "Ordem Cons",
            "item_pai",
            "Saldo",
            "Pedido",
            "Representante",
            "Entrega Pedido",
            "Local Prod.",
            "Familia",
            "Cliente",
            "Fábrica",
            "Ordem",
            "Data Abertura",
            "Baixa",
        ]
    ]
    ######## Calculos Baseados em estoque ##########
    estoque = sanitizar_dataframe(dfs.get("estoque"))
    consumo["estoque"] = (
        consumo["Item"].map(estoque.groupby("Item")["Qtde."].sum()).fillna(0)
    )

    CAMPOS_RAIZ = [
        "Cliente",
        "Fábrica",
        "Pedido",
        "Representante",
        "Entrega Pedido",
        "Data Abertura",
        "Saldo",
        "Baixa",
    ]

    itens_existentes = set(consumo["Item"].unique())
    mapa = consumo.drop_duplicates("Ordem Cons").set_index("Ordem Cons")
    mapa_item_ordem = consumo.drop_duplicates("Item").set_index("Item")["Ordem Cons"]

    raiz_rows = []
    for ordem_cons in consumo["Ordem Cons"]:
        visitados = set()
        atual = ordem_cons
        resultado = None

        while atual in mapa.index:
            if atual in visitados:
                break
            visitados.add(atual)
            linha = mapa.loc[atual]
            item_pai = linha["item_pai"]

            if pd.isna(item_pai) or item_pai not in itens_existentes:
                resultado = linha[CAMPOS_RAIZ]
                break

            if item_pai not in mapa_item_ordem.index:
                resultado = linha[CAMPOS_RAIZ]
                break

            atual = mapa_item_ordem[item_pai]

        raiz_rows.append(
            resultado.to_dict()
            if resultado is not None
            else {c: None for c in CAMPOS_RAIZ}
        )

    raiz_df = pd.DataFrame(raiz_rows, index=consumo.index)
    raiz_df.columns = [f"raiz_{c}" for c in raiz_df.columns]
    consumo = pd.concat([consumo, raiz_df], axis=1)

    csv_path = "CSV/"
    consumo.to_excel(csv_path + "consumo.xlsx", index=False)


calc_data(dfs)

In [ ]:
import pandas as pd
import networkx as nx

print(f"{dfs.keys()}\n")


def sanitizar_dataframe(df, limite=0.8):
    df = df.copy()

    for col in df.columns:
        serie = df[col].astype(str).str.strip()

        tentativa_data = pd.to_datetime(
            serie, errors="coerce", dayfirst=True, format="%d/%m/%Y"
        )
        if tentativa_data.notna().mean() > limite:
            df[col] = tentativa_data
            continue

        serie_num = serie.str.replace(".", "", regex=False).str.replace(
            ",", ".", regex=False
        )
        tentativa_num = pd.to_numeric(serie_num, errors="coerce")
        if tentativa_num.notna().mean() > limite:
            df[col] = tentativa_num
            continue

        df[col] = serie.replace({"": None})

    return df


def calc_data(dfs):

    ## Ajuste Ordens ##
    ordens = sanitizar_dataframe(dfs.get("ordens"))
    ordens = ordens[
        [
            "Cliente",
            "Fábrica",
            "Ordem",
            "Pedido",
            "Item",
            "Saldo",
            "Representante",
            "Entrega Pedido",
            "Data Abertura",
        ]
    ]
    ordens = ordens.rename(columns={"Ordem": "Ordem Prod", "Saldo": "Saldo Prod"})

    colunas = ["Entrega Pedido", "Data Abertura"]
    for col in colunas:
        # Remove o ponto e garante que a coluna seja tratada como string
        ordens[col] = (
            ordens[col]
            .astype(str)
            .str.strip()
            .str.replace(r"[^\d]", "", regex=True)  # remove tudo que não for número
            .pipe(lambda s: pd.to_datetime(s, format="%d%m%Y", errors="coerce"))
        )
    ## Ajuste Consumo ##
    consumo = sanitizar_dataframe(dfs.get("cons"))

    consumo["Item"] = consumo["Item"].str.split("-").str[0].str.strip()
    print(consumo.columns)
    consumo = consumo[
        ["Item", "Baixa", "Consumo", "Local Prod.", "OP", "Familia", "Den. Item"]
    ]
    consumo = consumo.rename(columns={"OP": "Ordem Cons"})

    ######## Item Pai ##########

    consumo["item_pai"] = consumo["Ordem Cons"].map(
        ordens.set_index("Ordem Prod")["Item"]
    )
    ######## Importa dados ##########
    consumo = consumo.merge(ordens[["Item", "Ordem Prod"]], on="Item", how="left")

    consumo = consumo.merge(
        ordens[
            [
                "Cliente",
                "Fábrica",
                "Ordem Prod",
                "Pedido",
                "Saldo Prod",
                "Representante",
                "Entrega Pedido",
                "Data Abertura",
            ]
        ].rename(columns={"Ordem Prod": "Ordem", "Saldo Prod": "Saldo"}),
        left_on="Ordem Cons",
        right_on="Ordem",
        how="left",
    )

    ######## Ordena as colunas ##########

    consumo = consumo[
        [
            "Ordem Prod",
            "Item",
            "Consumo",
            "Ordem Cons",
            "item_pai",
            "Saldo",
            "Pedido",
            "Representante",
            "Entrega Pedido",
            "Local Prod.",
            "Familia",
            "Cliente",
            "Fábrica",
            "Ordem",
            "Data Abertura",
            "Baixa",
            "Den. Item",
        ]
    ]
    ######## Calculos Baseados em estoque ##########
    estoque = sanitizar_dataframe(dfs.get("estoque"))
    consumo["estoque"] = (
        consumo["Item"].map(estoque.groupby("Item")["Qtde."].sum()).fillna(0)
    )

    from tqdm import tqdm
    import networkx as nx

    ######## Propagação dos atributos da raiz via NetworkX ##########

    CAMPOS_RAIZ = [
        "Cliente",
        "Fábrica",
        "Pedido",
        "Representante",
        "Entrega Pedido",
        "Data Abertura",
        "Saldo",
        "Baixa",
    ]

    itens_existentes = set(consumo["Item"].unique())

    arestas = consumo[["item_pai", "Item", "Ordem Cons"]].drop_duplicates()
    G = nx.DiGraph()
    G.add_edges_from(
        (row["item_pai"], row["Item"], {"ordem_cons": row["Ordem Cons"]})
        for _, row in arestas.iterrows()
    )

    raizes = [n for n in G.nodes if G.in_degree(n) == 0 and n not in itens_existentes]

    mapa_attrs = (
        consumo[consumo["item_pai"].isin(raizes)]
        .drop_duplicates("item_pai")
        .set_index("item_pai")[CAMPOS_RAIZ]
        .to_dict(orient="index")
    )

    mapa_no = {}
    for raiz in tqdm(raizes, desc="Propagando raízes", unit="raiz"):
        if raiz not in mapa_attrs:
            continue
        attrs = mapa_attrs[raiz]
        mapa_no[raiz] = attrs
        for descendente in nx.descendants(G, raiz):
            mapa_no[descendente] = attrs

    raiz_df = (
        consumo["item_pai"]
        .map(mapa_no)
        .apply(lambda x: x if isinstance(x, dict) else {c: None for c in CAMPOS_RAIZ})
    )
    raiz_df = pd.DataFrame(raiz_df.tolist(), index=consumo.index)
    raiz_df.columns = [f"raiz_{c}" for c in CAMPOS_RAIZ]
    consumo = pd.concat([consumo, raiz_df], axis=1)


    #### Gera Excel ####
    print ("gerar excel")
    csv_path = "CSV/"
    consumo.to_excel(csv_path + "consumo.xlsx", index=False)


calc_data(dfs)

dict_keys(['ordens', 'apoio_compras', 'cons', 'estoque'])

Index(['Tipo', 'Grupo', 'Item', 'Local Estoque', 'Baixa', 'Situação',
       'Den. Item', 'Consumo', 'Local Prod.', 'OP', 'Familia', 'Família'],
      dtype='str')


Propagando raízes: 100%|██████████| 5576/5576 [00:00<00:00, 78715.90raiz/s]
